# Union Mask and Binary-Mutation Clustering Analysis
Workflow completo per: caricamento dati, costruzione di `X_bin`, clustering nello spazio originale, visualizzazioni PCA e analisi di qualita dei cluster.

In [ ]:
import adabmDCA
import copy
import time
import os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# ===============================================================================
# Import adabmDCA utility functions
# ===============================================================================
# These functions provide core functionality for:
# - FASTA file I/O and sequence handling
# - Statistical analysis (frequencies, correlations)
# - One-hot encoding of sequences
# - Parameter and chain initialization
# - MCMC sampling
# - Energy computations
# ===============================================================================

from adabmDCA.fasta import get_tokens, import_from_fasta, compute_weights
from adabmDCA.stats import get_freq_single_point, get_freq_two_points
from adabmDCA.functional import one_hot
from adabmDCA.utils import init_parameters, init_chains, get_device, get_dtype
from adabmDCA.sampling import get_sampler

from adabmDCA.io import load_params, load_chains
# from adabmDCA.resampling import compute_mixing_time, compute_seqID, get_mean_seqid
from adabmDCA.utils import resample_sequences
from adabmDCA.statmech import compute_energy

# ===============================================================================
# Configure computational device and data type
# ===============================================================================
device = get_device("cuda")      # Use GPU for accelerated computation
dtype = get_dtype("float32")     # Use 32-bit floating point precision


## 1) Data Loading
Caricamento dei dataset `DN`, `DT-`, `DT+` e definizione dei token RNA.

In [ ]:
tokens = get_tokens("rna")  # Standard protein alphabet (20 amino acids + gap)
headers_DN, data_DN = import_from_fasta(
    "Group_I_intron-2/DN&DT/DN.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
headers_DTm, data_DTm = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT-.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
headers_DTp, data_DTp = import_from_fasta(
    "Group_I_intron-2/DN&DT/DT+.fasta", 
    tokens=tokens, 
    filter_sequences=True  # Remove problematic sequences
)
# output_path = "../images/"

## 2) Union-Mask Clustering Function
Definizione della funzione `union_mask_clustering` (approccio greedy agglomerativo).

In [ ]:
import heapq
import numpy as np


def union_mask_clustering(
    data: np.ndarray,
    tau: float = 0.30,
    show_initial_hist: bool = False,
    verbose: bool = False,
    show_merge_tree: bool = False,
    tree_max_positions: int = 20,
) -> list[list[int]]:
    """
    Greedy agglomerative Union Mask Clustering.

    Definitions
    -----------
    - Union mask of a cluster C: columns where at least one sequence in C differs.
    - m(C): size of the union mask.
    - Directed merge cost: cost(C_i, C_j) = m(C_i U C_j) - m(C_i).

    Stopping rule
    -------------
    Stop when the minimum valid directed merge cost is strictly greater than tau * L.

    Notes on implementation
    -----------------------
    - We keep a min-heap of directed candidate merges.
    - Heap entries can become stale after merges; stale entries are lazily discarded.
    - If a popped entry is alive but has outdated cost/tie-break key, we recompute and
      push the updated value back. This preserves exact greedy behavior.
    - Optionally, we build and print a merge tree (or forest if multiple roots remain).
    """
    # ------------------------------------------------------------------
    # Input validation
    # ------------------------------------------------------------------
    data = np.asarray(data)
    if data.ndim != 2:
        raise ValueError("`data` must be a 2D array with shape (N, L).")
    if not (0.0 <= tau <= 1.0):
        raise ValueError("`tau` must be in [0, 1].")

    N, L = data.shape
    threshold = tau * L

    # ------------------------------------------------------------------
    # Cluster state (one singleton per row at initialization)
    # ------------------------------------------------------------------
    members: dict[int, list[int]] = {i: [i] for i in range(N)}

    # For consistency with the formal definition we explicitly store union masks and m(C).
    masks: dict[int, np.ndarray] = {i: np.zeros(L, dtype=bool) for i in range(N)}
    mask_size: dict[int, int] = {i: 0 for i in range(N)}

    # Per-cluster extrema let us compute m(C_i U C_j) exactly and fast.
    mins: dict[int, np.ndarray] = {i: data[i].copy() for i in range(N)}
    maxs: dict[int, np.ndarray] = {i: data[i].copy() for i in range(N)}

    # Active cluster ids.
    alive: set[int] = set(range(N))

    # Monotone id for newly created merged clusters.
    next_id = N

    # Version counters to invalidate stale heap entries after any cluster update/removal.
    versions: dict[int, int] = {i: 0 for i in range(N)}

    # Tree structure: each node stores children and merge cost.
    # Leaves (singletons) have left/right = None and cost = 0.
    tree: dict[int, dict[str, object]] = {
        i: {"left": None, "right": None, "cost": 0}
        for i in range(N)
    }

    # ------------------------------------------------------------------
    # Directed merge cost: C_j into C_i
    # ------------------------------------------------------------------
    def merge_cost(cid_i: int, cid_j: int) -> int:
        merged_min = np.minimum(mins[cid_i], mins[cid_j])
        merged_max = np.maximum(maxs[cid_i], maxs[cid_j])
        merged_size = int(np.count_nonzero(merged_min != merged_max))
        cost = merged_size - mask_size[cid_i]
        if not (0 <= cost <= L):
            raise RuntimeError(
                f"Invalid merge cost {cost} for ({cid_i} -> {cid_j}); expected in [0, {L}]"
            )
        return cost

    # ------------------------------------------------------------------
    # Initial heap build with all directed pairs.
    # Heap key: (cost, m(C_i), C_i, C_j, version_i, version_j)
    # The second key enforces the required tie-breaker on smallest base mask size.
    # ------------------------------------------------------------------
    heap: list[tuple[int, int, int, int, int, int]] = []
    alive_list = list(alive)

    for a_idx, cid_a in enumerate(alive_list):
        for cid_b in alive_list[a_idx + 1 :]:
            cost_ab = merge_cost(cid_a, cid_b)
            cost_ba = merge_cost(cid_b, cid_a)

            heapq.heappush(
                heap,
                (cost_ab, mask_size[cid_a], cid_a, cid_b, versions[cid_a], versions[cid_b]),
            )
            heapq.heappush(
                heap,
                (cost_ba, mask_size[cid_b], cid_b, cid_a, versions[cid_b], versions[cid_a]),
            )

    if show_initial_hist and heap:
        initial_costs = [entry[0] for entry in heap]
        plt.figure(figsize=(8, 5))
        plt.hist(initial_costs, bins=50, color="skyblue", edgecolor="black")
        plt.title("Initial Merge Cost Distribution")
        plt.xlabel("Merge Cost (Increase in Mask Size)")
        plt.ylabel("Frequency")
        plt.grid(axis="y", alpha=0.75)
        plt.show()

    # ------------------------------------------------------------------
    # Helper: print the 10 lowest costs currently in the heap.
    # Uses nsmallest which does NOT modify the heap.
    # ------------------------------------------------------------------
    def print_top_costs(n: int = 10) -> None:
        top = heapq.nsmallest(n, heap)
        costs = [entry[0] for entry in top]
        print(f"  Top-{len(costs)} heap costs (before pop): {costs}")

    def format_positions(mask: np.ndarray) -> str:
        # 1-based positions for readability
        pos = (np.flatnonzero(mask) + 1).tolist()
        if len(pos) <= tree_max_positions:
            return str(pos)
        head = pos[:tree_max_positions]
        return f"{head} ... (+{len(pos) - tree_max_positions} others)"

    def print_tree(node_id: int, prefix: str = "", is_last: bool = True) -> None:
        node = tree[node_id]
        connector = "└── " if is_last else "├── "

        positions_str = format_positions(masks[node_id])
        print(
            f"{prefix}{connector}Node {node_id} | cost={node['cost']} | "
            f"n_seq={len(members[node_id])} | mask_size={mask_size[node_id]} | "
            f"mask_pos={positions_str}"
        )

        left = node["left"]
        right = node["right"]
        if left is None and right is None:
            return

        child_prefix = prefix + ("    " if is_last else "│   ")
        children = [left, right]
        for idx, child in enumerate(children):
            print_tree(child, child_prefix, idx == len(children) - 1)

    # ------------------------------------------------------------------
    # Greedy merge loop
    # ------------------------------------------------------------------
    while heap:
        # Show the 10 lowest costs in the heap before popping (includes stale entries).
        if verbose:
            print_top_costs(10)

        cost, base_m, cid_i, cid_j, vi, vj = heapq.heappop(heap)

        # 1) Discard entries referring to inactive clusters.
        if cid_i not in alive or cid_j not in alive:
            if verbose:
                print(
                    f"Discarding: one of the clusters {cid_i} or {cid_j} is no longer active."
                )
            continue

        # 2) Discard entries with outdated versions (stale after prior merges).
        if versions[cid_i] != vi or versions[cid_j] != vj:
            continue

        # 3) Recompute exact current key for this directed pair.
        current_cost = merge_cost(cid_i, cid_j)
        current_base_m = mask_size[cid_i]

        # If stale by value (cost or tie-break key changed), refresh and continue.
        if current_cost != cost or current_base_m != base_m:
            heapq.heappush(
                heap,
                (
                    current_cost,
                    current_base_m,
                    cid_i,
                    cid_j,
                    versions[cid_i],
                    versions[cid_j],
                ),
            )
            continue

        # This is the minimum valid directed merge candidate for this iteration.
        if cost > threshold:
            if verbose:
                print(
                    f"Stopping: minimum merge cost {cost} > tau * L = {threshold:.3f}"
                )
            break

        if verbose:
            print(
                f"Merge: {cid_i} absorbs {cid_j} | "
                f"cost={cost}, m(base)={mask_size[cid_i]}, alive={len(alive)}"
            )

        # ------------------------------------------------------------------
        # Execute merge and create a new cluster id.
        # ------------------------------------------------------------------
        new_id = next_id
        next_id += 1

        members[new_id] = members[cid_i] + members[cid_j]
        mins[new_id] = np.minimum(mins[cid_i], mins[cid_j])
        maxs[new_id] = np.maximum(maxs[cid_i], maxs[cid_j])

        # Store merge in the tree.
        tree[new_id] = {
            "left": cid_i,
            "right": cid_j,
            "cost": int(cost),
        }

        # New union mask and m(C) from extrema profile.
        new_mask = mins[new_id] != maxs[new_id]
        masks[new_id] = new_mask
        mask_size[new_id] = int(np.count_nonzero(new_mask))

        # Activate new cluster.
        alive.remove(cid_i)
        alive.remove(cid_j)
        alive.add(new_id)

        # Invalidate old entries that mention cid_i/cid_j.
        versions[cid_i] += 1
        versions[cid_j] += 1

        # Initialize version for the new cluster.
        versions[new_id] = 0

        # ------------------------------------------------------------------
        # Add new directed candidates involving new_id and every other alive cluster.
        # ------------------------------------------------------------------
        for other in alive:
            if other == new_id:
                continue

            c_new_other = merge_cost(new_id, other)
            c_other_new = merge_cost(other, new_id)

            heapq.heappush(
                heap,
                (
                    c_new_other,
                    mask_size[new_id],
                    new_id,
                    other,
                    versions[new_id],
                    versions[other],
                ),
            )
            heapq.heappush(
                heap,
                (
                    c_other_new,
                    mask_size[other],
                    other,
                    new_id,
                    versions[other],
                    versions[new_id],
                ),
            )

    if show_merge_tree:
        print("\n=== Merge tree (forest) ===")
        roots = sorted(alive)
        for idx, root in enumerate(roots, start=1):
            print(f"Root {idx}: cluster {root}")
            print_tree(root)

    # Return active clusters (deterministic order by cluster id).
    return [members[cid] for cid in sorted(alive)]

## 3) Subsampling and Initial Setup
Preparazione del campione `DT_subsample` per le analisi successive.

In [ ]:
rna_vector = np.array([
    0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 
    1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 
    0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 
    0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 
    1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 
    1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 
    0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 
    1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 
    0, 1, 1, 0, 0
])

rna_vector = rna_vector[3:-1]
print("Lunghezza array:", len(rna_vector))


plt.bar(range(len(rna_vector)), 1- rna_vector)
plt.show()

In [ ]:
DT = np.concatenate((data_DTm, data_DTp), axis=0)

# Use the full DT dataset (no subsampling), preserving order: first DTm then DTp.
DT_full = np.concatenate((data_DTm, data_DTp), axis=0)
n_dtm = data_DTm.shape[0]
n_dtp = data_DTp.shape[0]



rng = np.random.default_rng()
idx = rng.choice(DT.shape[0], size=100, replace=False)
DT_subsample = DT[idx, :]
print(f"DT_subsample shape: {DT_subsample.shape}")

In [ ]:
print(f"Red: {n_dtm / DT.shape[0]:.3f}, Green: {n_dtp / DT.shape[0]:.3f}")

### 3.1) Optional Union-Mask Run (Disabled)
Questa sezione e opzionale: il blocco operativo di `union_mask_clustering` e stato mantenuto disattivato per non appesantire il run principale.

## 4) Binary Mutation Encoding and Clustering (Original Space)
Costruzione di `X_bin` rispetto alla reference e clustering K-means nello spazio binario originale.

In [ ]:
# ===============================================================================
# MUTATION BINARY ENCODING VS REFERENCE + PCA + K-MEANS
# ===============================================================================
# For each sequence in DT_full, create a binary vector where:
# - 1 means mutated vs reference at that position
# - 0 means identical to reference

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

reference_seq = data_DN[0]

if DT_full.shape[1] != reference_seq.shape[0]:
    raise ValueError(
        f"Length mismatch: DT_full has L={DT_full.shape[1]}, "
        f"reference has L={reference_seq.shape[0]}"
    )

# Binary mutation matrix: shape (N_sequences, L_positions)
X_bin = (DT_full != reference_seq).astype(np.uint8)

print(f"Binary mutation matrix shape: {X_bin.shape}")
print(f"Mean number of mutations per sequence: {X_bin.sum(axis=1).mean():.2f}")

# -----------------------------
# PCA only for visualization
# -----------------------------
pca = PCA(n_components=2, random_state=None)
X_pca = pca.fit_transform(X_bin)

expl_var = pca.explained_variance_ratio_
print(f"Explained variance (PC1, PC2): {expl_var[0]:.4f}, {expl_var[1]:.4f}")
print(f"Cumulative explained variance (2 PCs): {expl_var.sum():.4f}")

# -----------------------------
# K-means clustering on ORIGINAL binary space (X_bin)
# -----------------------------
k_values = range(2, 20)
sil_scores = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=None, n_init=20)
    labels_k = km.fit_predict(X_bin)
    sil_scores.append(silhouette_score(X_bin, labels_k))

best_k = list(k_values)[int(np.argmax(sil_scores))]
print(f"Best k by silhouette (on original binary space): {best_k}")

kmeans = KMeans(n_clusters=best_k, random_state=None, n_init=20)
labels = kmeans.fit_predict(X_bin)

unique_labels, counts = np.unique(labels, return_counts=True)
print("Cluster sizes:")
for lbl, cnt in zip(unique_labels, counts):
    print(f"  Cluster {lbl}: {cnt} sequences")

# -----------------------------
# Visualization (PCA plane, labels from X_bin clustering)
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

axes[0].plot(list(k_values), sil_scores, marker="o", color="black")
axes[0].set_title("Silhouette score vs k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette score")
axes[0].grid(True, alpha=0.3)

scatter = axes[1].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    cmap="tab10",
    s=20,
    alpha=0.8,
    edgecolor="none",
)
axes[1].set_title(f"PCA (2D) + K-means on original space (k={best_k})")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(True, alpha=0.3)
legend1 = axes[1].legend(*scatter.legend_elements(), title="Cluster", loc="best")
axes[1].add_artist(legend1)

plt.show()

# Keep outputs available for downstream cells
mutation_binary_dataset = X_bin
pca_embedding = X_pca
kmeans_labels = labels
kmeans_model_original_space = kmeans

In [ ]:
best_k = 9
kmeans = KMeans(n_clusters=best_k, n_init=20) #random_state=None
labels = kmeans.fit_predict(X_bin)

unique_labels, counts = np.unique(labels, return_counts=True)
print("Cluster sizes:")
for lbl, cnt in zip(unique_labels, counts):
    print(f"  Cluster {lbl}: {cnt} sequences")

# -----------------------------
# Visualization
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

axes[0].plot(list(k_values), sil_scores, marker="o", color="black")
axes[0].set_title("Silhouette score vs k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette score")
axes[0].grid(True, alpha=0.3)

scatter = axes[1].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    cmap="tab10",
    s=20,
    alpha=0.8,
    edgecolor="none",
)
axes[1].set_title(f"PCA (2D) + K-means on original space (k={best_k})")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(True, alpha=0.3)
legend1 = axes[1].legend(*scatter.legend_elements(), title="Cluster", loc="best")
axes[1].add_artist(legend1)

plt.show()

# Keep outputs available for downstream cells
mutation_binary_dataset = X_bin
pca_embedding = X_pca
kmeans_labels = labels
kmeans_model_original_space = kmeans

## 5) PCA Projections and Cluster Visual Checks
Visualizzazioni nel piano PCA (etichette cluster e colorazioni alternative).

In [ ]:
# ===============================================================================
# PCA PAIR PLOTS: PCx vs PCy for x,y in {1,2,3,4}
# ===============================================================================
# This cell creates:
# 1) A single PC1 vs PC2 figure (saved as PNG)
# 2) A 6-subplot figure with all unique pairs among the first 4 principal components

from itertools import combinations
from sklearn.decomposition import PCA

# Reuse the binary mutation dataset if already available; otherwise rebuild it.
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

pca4 = PCA(n_components=4, random_state=None)
X_pca4 = pca4.fit_transform(X_bin_local)

# Color by kmeans labels if available, otherwise single color.
use_labels = "kmeans_labels" in globals() and len(kmeans_labels) == X_pca4.shape[0]

# -------------------------------------------------------------------------------
# Figure 1: only PC1 vs PC2 (saved as PNG)
# -------------------------------------------------------------------------------
fig_pc12, ax_pc12 = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)

if use_labels:
    sc_pc12 = ax_pc12.scatter(
        X_pca4[:, 0],
        X_pca4[:, 1],
        c=kmeans_labels,
        cmap="tab10",
        s=18,
        alpha=0.8,
        edgecolor="none",
    )
    legend_pc12 = ax_pc12.legend(*sc_pc12.legend_elements(), title="Cluster", loc="best")
    ax_pc12.add_artist(legend_pc12)
else:
    ax_pc12.scatter(
        X_pca4[:, 0],
        X_pca4[:, 1],
        color="steelblue",
        s=18,
        alpha=0.8,
        edgecolor="none",
    )

ax_pc12.set_xlabel("PC1")
ax_pc12.set_ylabel("PC2")
ax_pc12.set_title("PC1 vs PC2")
ax_pc12.grid(True, alpha=0.3)

pc12_png_path = "pca_pc1_vs_pc2.png"
fig_pc12.savefig(pc12_png_path, dpi=300, bbox_inches="tight")
print(f"Saved PNG: {pc12_png_path}")
plt.show()

# -------------------------------------------------------------------------------
# Figure 2: existing 6-subplot layout (not saved)
# -------------------------------------------------------------------------------
pairs = list(combinations(range(4), 2))  # (0,1), (0,2), ..., (2,3)

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    if use_labels:
        sc = ax.scatter(
            X_pca4[:, i],
            X_pca4[:, j],
            c=kmeans_labels,
            cmap="tab10",
            s=16,
            alpha=0.8,
            edgecolor="none",
        )
    else:
        sc = ax.scatter(
            X_pca4[:, i],
            X_pca4[:, j],
            color="steelblue",
            s=16,
            alpha=0.8,
            edgecolor="none",
        )

    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.grid(True, alpha=0.3)

if use_labels:
    handles, labels_legend = sc.legend_elements()
    fig.legend(handles, labels_legend, title="Cluster", loc="upper right")

exp = pca4.explained_variance_ratio_
fig.suptitle(
    f"Pairwise PCA projections (PC1-PC4) | explained variance: "
    f"PC1={exp[0]:.3f}, PC2={exp[1]:.3f}, PC3={exp[2]:.3f}, PC4={exp[3]:.3f}",
    fontsize=12,
)

plt.show()

# Optional export for downstream analysis
pca4_embedding = X_pca4
pca4_model = pca4

In [ ]:
# ===============================================================================
# PCA PAIR PLOTS (PC1-PC4) COLORED BY TOTAL MUTATION COUNT
# ===============================================================================
# This cell creates:
# 1) A single PC1 vs PC2 figure (saved as PNG)
# 2) A 6-subplot figure with all unique pairs among the first 4 principal components

from itertools import combinations
from sklearn.decomposition import PCA

# Reuse the binary mutation dataset if available; otherwise rebuild it.
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

# Value used for gradient color: total number of mutated positions per sequence.
mutation_load = X_bin_local.sum(axis=1)

pca4 = PCA(n_components=4, random_state=None)
X_pca4 = pca4.fit_transform(X_bin_local)
pairs = list(combinations(range(4), 2))

# -------------------------------------------------------------------------------
# Figure 1: only PC1 vs PC2 (saved as PNG)
# -------------------------------------------------------------------------------
fig_pc12, ax_pc12 = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)
sc_pc12 = ax_pc12.scatter(
    X_pca4[:, 0],
    X_pca4[:, 1],
    c=mutation_load,
    cmap="viridis",
    s=18,
    alpha=0.85,
    edgecolor="none",
)
ax_pc12.set_xlabel("PC1")
ax_pc12.set_ylabel("PC2")
ax_pc12.set_title("PC1 vs PC2 | color = mutation load")
ax_pc12.grid(True, alpha=0.3)
cbar_pc12 = fig_pc12.colorbar(sc_pc12, ax=ax_pc12, fraction=0.046, pad=0.04)
cbar_pc12.set_label("Total mutated positions per sequence")

pc12_png_path = "pca_pc1_vs_pc2_mutation_load.png"
fig_pc12.savefig(pc12_png_path, dpi=300, bbox_inches="tight")
print(f"Saved PNG: {pc12_png_path}")
plt.show()

# -------------------------------------------------------------------------------
# Figure 2: existing 6-subplot layout (not saved)
# -------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

mappable = None
for ax, (i, j) in zip(axes, pairs):
    mappable = ax.scatter(
        X_pca4[:, i],
        X_pca4[:, j],
        c=mutation_load,
        cmap="viridis",
        s=16,
        alpha=0.85,
        edgecolor="none",
    )
    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.grid(True, alpha=0.3)

cbar = fig.colorbar(mappable, ax=axes.tolist(), fraction=0.02, pad=0.02)
cbar.set_label("Total mutated positions per sequence", fontsize=11)

exp = pca4.explained_variance_ratio_
fig.suptitle(
    f"Pairwise PCA projections (PC1-PC4) | color = mutation load | "
    f"PC1={exp[0]:.3f}, PC2={exp[1]:.3f}, PC3={exp[2]:.3f}, PC4={exp[3]:.3f}",
    fontsize=12,
)

plt.show()

# Optional export for downstream analysis
pca4_embedding_gradient = X_pca4
mutation_load_per_sequence = mutation_load

In [ ]:
# ===============================================================================
# PCA PAIR PLOTS (PC1-PC4) ON FULL DT, COLORED BY ORIGIN: DTm (red) vs DTp (green)
# ===============================================================================

from itertools import combinations
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D

reference_seq = data_DN[0]
if DT_full.shape[1] != reference_seq.shape[0]:
    raise ValueError(
        f"Length mismatch: DT_full has L={DT_full.shape[1]}, reference has L={reference_seq.shape[0]}"
    )

X_bin_full = (DT_full != reference_seq).astype(np.uint8)

# Origin labels: first block DTm, second block DTp.
is_dtm = np.zeros(DT_full.shape[0], dtype=bool)
is_dtm[:n_dtm] = True
is_dtp = ~is_dtm

pca4_full = PCA(n_components=4, random_state=None)
X_pca4_full = pca4_full.fit_transform(X_bin_full)
pairs = list(combinations(range(4), 2))

# -------------------------------------------------------------------------------
# Figure 1: only PC1 vs PC2 (saved as PNG)
# -------------------------------------------------------------------------------
fig_pc12, ax_pc12 = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)
ax_pc12.scatter(
    X_pca4_full[is_dtm, 0],
    X_pca4_full[is_dtm, 1],
    color="red",
    s=14,
    alpha=0.65,
    edgecolor="none",
)
ax_pc12.scatter(
    X_pca4_full[is_dtp, 0],
    X_pca4_full[is_dtp, 1],
    color="green",
    s=14,
    alpha=0.65,
    edgecolor="none",
)
ax_pc12.set_xlabel("PC1")
ax_pc12.set_ylabel("PC2")
ax_pc12.set_title("PC1 vs PC2")
ax_pc12.grid(True, alpha=0.3)

legend_handles_pc12 = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="Green"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="Red"),
]
ax_pc12.legend(handles=legend_handles_pc12, title="Origin", loc="best")

pc12_png_path = "pca_pc1_vs_pc2_origin.png"
fig_pc12.savefig(pc12_png_path, dpi=300, bbox_inches="tight")
print(f"Saved PNG: {pc12_png_path}")
plt.show()

# -------------------------------------------------------------------------------
# Figure 2: existing 6-subplot layout (not saved)
# -------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(
        X_pca4_full[is_dtm, i],
        X_pca4_full[is_dtm, j],
        color="red",
        s=14,
        alpha=0.65,
        edgecolor="none",
        label="DTm",
    )
    ax.scatter(
        X_pca4_full[is_dtp, i],
        X_pca4_full[is_dtp, j],
        color="green",
        s=14,
        alpha=0.65,
        edgecolor="none",
        label="DTp",
    )
    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.grid(True, alpha=0.3)

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="DTm"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="DTp"),
]
fig.legend(handles=legend_handles, title="Origin", loc="upper right")

exp_full = pca4_full.explained_variance_ratio_
fig.suptitle(
    "Pairwise PCA projections (PC1-PC4) | colors: DTm=red, DTp=green | "
    f"PC1={exp_full[0]:.3f}, PC2={exp_full[1]:.3f}, PC3={exp_full[2]:.3f}, PC4={exp_full[3]:.3f}",
    fontsize=12,
)

plt.show()

# Optional export for downstream analysis
pca4_full_embedding = X_pca4_full
pca4_full_model = pca4_full
full_origin_is_dtm = is_dtm

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(
        X_pca4_full[is_dtm, i],
        X_pca4_full[is_dtm, j],
        color="red",
        s=14,
        alpha=0.65,
        edgecolor="none",
        label="DTm",
    )
    # ax.scatter(
    #     X_pca4_full[is_dtp, i],
    #     X_pca4_full[is_dtp, j],
    #     color="green",
    #     s=14,
    #     alpha=0.65,
    #     edgecolor="none",
    #     label="DTp",
    # )
    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.grid(True, alpha=0.3)

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="DTm"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="DTp"),
]
fig.legend(handles=legend_handles, title="Origin", loc="upper right")

exp_full = pca4_full.explained_variance_ratio_
fig.suptitle(
    "Pairwise PCA projections (PC1-PC4) | colors: DTm=red, DTp=green | "
    f"PC1={exp_full[0]:.3f}, PC2={exp_full[1]:.3f}, PC3={exp_full[2]:.3f}, PC4={exp_full[3]:.3f}",
    fontsize=12,
)

plt.show()






fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    # ax.scatter(
    #     X_pca4_full[is_dtm, i],
    #     X_pca4_full[is_dtm, j],
    #     color="red",
    #     s=14,
    #     alpha=0.65,
    #     edgecolor="none",
    #     label="DTm",
    # )
    ax.scatter(
        X_pca4_full[is_dtp, i],
        X_pca4_full[is_dtp, j],
        color="green",
        s=14,
        alpha=0.65,
        edgecolor="none",
        label="DTp",
    )
    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.grid(True, alpha=0.3)

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="DTm"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="DTp"),
]
fig.legend(handles=legend_handles, title="Origin", loc="upper right")

exp_full = pca4_full.explained_variance_ratio_
fig.suptitle(
    "Pairwise PCA projections (PC1-PC4) | colors: DTm=red, DTp=green | "
    f"PC1={exp_full[0]:.3f}, PC2={exp_full[1]:.3f}, PC3={exp_full[2]:.3f}, PC4={exp_full[3]:.3f}",
    fontsize=12,
)

plt.show()


## 6) Cluster-Wise Quantitative Analysis
Metriche per cluster e profili di mutazione per posizione (normalizzati).

In [ ]:
# ===============================================================================
# CLUSTER ANALYSIS ON ORIGINAL BINARY SPACE (X_bin)
# 1) Mean intra-cluster distance among X_bin
# 2) Mean distance from reference per cluster (sum of X_bin)
# 3) One subplot per cluster: normalized mutation frequency per position
# ===============================================================================

# Retrieve binary dataset and labels from previous cells.
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
elif "X_bin" in globals():
    X_bin_local = X_bin
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

if "kmeans_labels" in globals():
    labels_local = np.asarray(kmeans_labels)
else:
    raise ValueError("kmeans_labels not found. Run cell 6 (or cell 7/8 workflow) first.")

if X_bin_local.shape[0] != labels_local.shape[0]:
    raise ValueError(
        f"Size mismatch: X_bin has {X_bin_local.shape[0]} rows but labels has {labels_local.shape[0]} entries."
    )

L = X_bin_local.shape[1]
cluster_ids = np.unique(labels_local)
n_dtm_local = int(n_dtm) if "n_dtm" in globals() else data_DTm.shape[0]

# -------------------------------------------------------------------------------
# 1) Mean intra-cluster pairwise distance in original binary space
# We use Hamming count (number of differing positions).
# For efficiency, average pairwise Hamming is computed without O(n^2) matrices:
# sum_j p_j*(n-p_j) / C(n,2), where p_j = ones at position j.
# -------------------------------------------------------------------------------
analysis_rows = []
for cid in cluster_ids:
    idx = np.where(labels_local == cid)[0]
    Xc = X_bin_local[idx]
    n_c = Xc.shape[0]

    if n_c >= 2:
        ones_per_pos = Xc.sum(axis=0).astype(np.float64)
        total_pairwise_hamming = np.sum(ones_per_pos * (n_c - ones_per_pos))
        mean_intra_distance = total_pairwise_hamming / (n_c * (n_c - 1) / 2.0)
    else:
        mean_intra_distance = 0.0

    # 2) Mean distance from reference in cluster: mean(sum(X_bin row))
    mean_ref_distance = Xc.sum(axis=1).mean()

    # add also fractions of DTm vs DTp in each cluster
    dtp_frac = np.mean(idx >= n_dtm_local) if idx.size > 0 else np.nan
    dtm_frac = 1 - dtp_frac

    analysis_rows.append((int(cid), int(n_c), float(mean_intra_distance), float(mean_ref_distance), dtm_frac, dtp_frac))

print("Cluster metrics (original space):")
print("cluster | size | mean intra-cluster distance | mean distance from reference || Green fraction")
for cid, n_c, intra_d, ref_d, dtm_frac, dtp_frac in analysis_rows:
    print(f"{cid:7d} | {n_c:4d} | {intra_d:27.3f} | {ref_d:28.3f} || {dtp_frac * 100:.3f}%")

# -------------------------------------------------------------------------------
# 3) Figure: one subplot per cluster
# x-axis: position (1..L), y-axis: normalized ones frequency in that cluster
# -------------------------------------------------------------------------------
n_clusters = len(cluster_ids)
n_cols = min(3, n_clusters)
n_rows = int(np.ceil(n_clusters / n_cols))

# One consistent color for each cluster subplot.
cluster_cmap = plt.get_cmap("tab20", n_clusters)
cluster_colors = {cid: cluster_cmap(i) for i, cid in enumerate(cluster_ids)}

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(6 * n_cols, 3.8 * n_rows),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

axes = np.array(axes).reshape(-1)
x_pos = np.arange(1, L + 1)

for ax_i, cid in enumerate(cluster_ids):
    ax = axes[ax_i]
    idx = np.where(labels_local == cid)[0]
    Xc = X_bin_local[idx]
    counts_ones = Xc.sum(axis=0)
    freq_ones = counts_ones / Xc.shape[0]
    cluster_color = cluster_colors[cid]

    # pearson correlation between freq_ones and rna_vector
    corr = np.corrcoef(freq_ones, 1 - rna_vector)[0, 1]
    dtp_frac = np.mean(idx >= n_dtm_local) if idx.size > 0 else np.nan

    ax.plot(x_pos, freq_ones, color=cluster_color, linewidth=1.2)
    ax.fill_between(x_pos, freq_ones, color=cluster_color, alpha=0.18)
    ax.text(0.02, 0.95, f"corr with free sites: {corr:.3f}", transform=ax.transAxes, fontsize=15, verticalalignment="top")
    ax.text(0.02, 0.88, f"Green: {dtp_frac * 100:.3f}%", transform=ax.transAxes, fontsize=15, verticalalignment="top")
    # ax.text(0.02, 0.81, f"Red: {dtm_frac *100:.3f}%", transform=ax.transAxes, fontsize=10, verticalalignment="top")
    ax.bar(range(len(rna_vector)), 0.1 * (1 - rna_vector), color="black", alpha=1, label="free positions")
    ax.set_title(f"Cluster {cid} (n={Xc.shape[0]})")
    ax.set_xlabel("Position in X_bin")
    ax.set_ylabel("Frequency of ones")
    ax.grid(True, alpha=0.25)
    ax.set_xlim(1, L)
    ax.set_ylim(0, 1)

# Hide any extra empty axes
for j in range(n_clusters, len(axes)):
    axes[j].axis("off")

fig.suptitle("Per-cluster normalized mutation frequency by position", fontsize=13)
plt.savefig("cluster_mutation_profiles.png", dpi=300, bbox_inches="tight")
plt.show()

# Save metrics for downstream cells
cluster_analysis_summary = analysis_rows

## 7) Entropy-Oriented Bernoulli Clustering
Clustering hard-Bernoulli per spingere `freq_ones` verso valori vicini a 0 o 1.

In [ ]:
# ===============================================================================
# CLUSTERING FOR NEAR-BINARY POSITION FREQUENCIES (freq_ones close to 0/1)
# Hard EM with independent Bernoulli components on X_bin
# ===============================================================================

# Input binary matrix
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
elif "X_bin" in globals():
    X_bin_local = X_bin
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

X = X_bin_local.astype(np.float64)
N, L = X.shape

# Number of clusters (reuse best_k if available, else fallback)
K =  int(best_k) if "best_k" in globals() else 8

# Hyperparameters
max_iter = 200
n_init = 2
alpha = 1e-2          # small smoothing to avoid exact 0/1 numerical issues
eps = 1e-6
rng = np.random #.default_rng()

def cluster_entropy_bits(p):
    p = np.clip(p, eps, 1 - eps)
    h = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    return h

def fit_hard_bernoulli(X, K, max_iter=80, alpha=1e-2, rng=None):
    N, L = X.shape

    # Random initial labels with all clusters represented
    labels = np.arange(N) % K
    rng.shuffle(labels)

    prev_obj = -np.inf
    for _ in range(max_iter):
        # M-step: estimate per-cluster Bernoulli parameters
        p = np.zeros((K, L), dtype=np.float64)
        sizes = np.zeros(K, dtype=int)
        for k in range(K):
            idx = np.where(labels == k)[0]
            sizes[k] = len(idx)
            if sizes[k] > 0:
                s = X[idx].sum(axis=0)
                p[k] = (s + alpha) / (sizes[k] + 2 * alpha)
            else:
                # temporary placeholder; will be fixed after reassignment
                p[k] = 0.5

        p = np.clip(p, eps, 1 - eps)
        logp = np.log(p)
        log1mp = np.log(1 - p)

        # E-step: assign each sample to cluster with maximum Bernoulli log-likelihood
        # score(i, k) = sum_j x_ij log p_kj + (1-x_ij) log(1-p_kj)
        scores = X @ (logp - log1mp).T + np.sum(log1mp, axis=1)
        new_labels = np.argmax(scores, axis=1)

        # Repair empty clusters by moving worst-fit points from largest clusters
        for k in range(K):
            if np.any(new_labels == k):
                continue
            donor = np.argmax(np.bincount(new_labels, minlength=K))
            donor_idx = np.where(new_labels == donor)[0]
            donor_scores = scores[donor_idx, donor]
            move_idx = donor_idx[np.argmin(donor_scores)]
            new_labels[move_idx] = k

        obj = np.sum(scores[np.arange(N), new_labels])

        if np.array_equal(new_labels, labels) or abs(obj - prev_obj) < 1e-8:
            labels = new_labels
            break

        labels = new_labels
        prev_obj = obj

    # Final parameters and metrics
    p_final = np.zeros((K, L), dtype=np.float64)
    sizes_final = np.zeros(K, dtype=int)
    for k in range(K):
        idx = np.where(labels == k)[0]
        sizes_final[k] = len(idx)
        s = X[idx].sum(axis=0)
        p_final[k] = (s + alpha) / (sizes_final[k] + 2 * alpha)
    p_final = np.clip(p_final, eps, 1 - eps)

    # Weighted mean entropy per position (lower = frequencies closer to 0/1)
    H = cluster_entropy_bits(p_final)
    weighted_mean_entropy = np.average(H.mean(axis=1), weights=sizes_final)

    return labels, p_final, sizes_final, weighted_mean_entropy

best = None
for _ in range(n_init):
    labels_try, p_try, sizes_try, wH_try = fit_hard_bernoulli(
        X, K, max_iter=max_iter, alpha=alpha, rng=rng
    )
    # Keep model with smallest weighted entropy
    if (best is None) or (wH_try < best[3]):
        best = (labels_try, p_try, sizes_try, wH_try)

labels_pure, p_pure, sizes_pure, weighted_entropy = best

print(f"Hard-Bernoulli clustering completed with K={K}")
print(f"Weighted mean entropy per position: {weighted_entropy:.4f} bits (lower is better)")
print("Cluster sizes:")
for k, n_k in enumerate(sizes_pure):
    print(f"  Cluster {k}: {n_k} sequences")

# Compare with previous clustering frequencies if available
if "kmeans_labels" in globals() and len(kmeans_labels) == N:
    old_ids = np.unique(kmeans_labels)
    old_ent = []
    for cid in old_ids:
        Xc = X[np.where(kmeans_labels == cid)[0]]
        p_old = np.clip(Xc.mean(axis=0), eps, 1 - eps)
        old_ent.append(cluster_entropy_bits(p_old).mean())
    old_sizes = np.array([np.sum(kmeans_labels == cid) for cid in old_ids])
    old_weighted_entropy = np.average(old_ent, weights=old_sizes)
    print(f"Previous clustering weighted mean entropy: {old_weighted_entropy:.4f} bits")

# -------------------------------------------------------------------------------
# Plot normalized freq_ones per position for each new cluster
# -------------------------------------------------------------------------------
n_cols = min(3, K)
n_rows = int(np.ceil(K / n_cols))
x_pos = np.arange(1, L + 1)
n_dtm_local = int(n_dtm) if "n_dtm" in globals() else data_DTm.shape[0]

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(6 * n_cols, 3.8 * n_rows),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)
axes = np.array(axes).reshape(-1)

for k in range(K):
    ax = axes[k]
    freq_ones = p_pure[k]
    idx_k = np.where(labels_pure == k)[0]
    dtp_frac = np.mean(idx_k >= n_dtm_local) if idx_k.size > 0 else np.nan
    # pearson correlation between freq_ones and rna_vector
    corr = np.corrcoef(freq_ones, 1-rna_vector)[0, 1]
    ax.plot(x_pos, freq_ones, color="darkorange", linewidth=1.2)
    ax.fill_between(x_pos, freq_ones, color="darkorange", alpha=0.20)
    ax.text(0.02, 0.95, f"corr with free sites: {corr:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
    ax.text(0.02, 0.88, f"frac DTp: {dtp_frac:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
    ax.bar(range(len(rna_vector)), 0.1*(1- rna_vector), color="salmon", alpha=1, label="free positions")
    ax.set_title(f"Cluster {k} (n={sizes_pure[k]})")
    ax.set_xlabel("Position in X_bin")
    ax.set_ylabel("Frequency of ones")
    ax.set_xlim(1, L)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)

for j in range(K, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    "Hard-Bernoulli clustering: per-cluster freq_ones (target: near 0 or 1)",
    fontsize=13,
)
plt.show()

# Export results for downstream cells
pure_binary_labels = labels_pure
pure_binary_freq_ones = p_pure
pure_binary_cluster_sizes = sizes_pure
pure_binary_weighted_entropy = weighted_entropy

## 8) Automatic Selection of Number of Clusters
Ricerca di `K` con criteri informativi (BIC/AIC) e confronto con metrica di purezza.

## 9) t-SNE Embedding and Clustering in Embedded Space
Analisi con t-SNE, scelta automatica di `K` via silhouette e profili `ones_freq` per cluster.

In [ ]:
# ===============================================================================
# t-SNE + CLUSTERING + BEST K + PER-CLUSTER ones_freq PROFILES
# ===============================================================================

from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -------------------------------------------------------------------------------
# Prepare binary dataset X_bin
# -------------------------------------------------------------------------------
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
elif "X_bin" in globals():
    X_bin_local = X_bin
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

X = X_bin_local.astype(np.float64)
N, L = X.shape

# -------------------------------------------------------------------------------
# 1) t-SNE embedding (2D)
# -------------------------------------------------------------------------------
# Perplexity must be < N; keep a safe value for both small and large datasets.
perplexity = min(30.0, max(5.0, (N - 1) / 3.0))
if perplexity >= N:
    perplexity = max(2.0, N - 1.0)

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    random_state=None,
    max_iter=1000,
 )
X_tsne = tsne.fit_transform(X)

print(f"t-SNE embedding shape: {X_tsne.shape}")
print(f"t-SNE perplexity used: {perplexity:.2f}")

# -------------------------------------------------------------------------------
# 2) + 3) Clustering in t-SNE space and best-K selection
# -------------------------------------------------------------------------------
k_min = 2
k_max = min(20, max(2, N - 1))
k_values = range(k_min, k_max + 1)

sil_scores = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=None, n_init=20)
    lbl_k = km.fit_predict(X_tsne)
    sil_scores.append(silhouette_score(X_tsne, lbl_k))

best_k_tsne = list(k_values)[int(np.argmax(sil_scores))]
kmeans_tsne = KMeans(n_clusters=best_k_tsne, random_state=None, n_init=20)
labels_tsne = kmeans_tsne.fit_predict(X_tsne)

print(f"Best k on t-SNE space (silhouette): {best_k_tsne}")
u_lbl, u_cnt = np.unique(labels_tsne, return_counts=True)
print("Cluster sizes (t-SNE space):")
for lbl, cnt in zip(u_lbl, u_cnt):
    print(f"  Cluster {lbl}: {cnt} sequences")

# # -------------------------------------------------------------------------------
# # 4) Figure with n-cluster subplots: ones_freq profiles
# # -------------------------------------------------------------------------------
# cluster_ids = np.unique(labels_tsne)
# n_clusters = len(cluster_ids)
# n_cols = min(3, n_clusters)
# n_rows = int(np.ceil(n_clusters / n_cols))
# x_pos = np.arange(1, L + 1)

# fig, axes = plt.subplots(
#     n_rows,
#     n_cols,
#     figsize=(6 * n_cols, 3.8 * n_rows),
#     constrained_layout=True,
#     sharex=True,
#     sharey=True,
#  )
# axes = np.array(axes).reshape(-1)

# for ax_i, cid in enumerate(cluster_ids):
#     ax = axes[ax_i]
#     idx = np.where(labels_tsne == cid)[0]
#     Xc = X[idx]
#     ones_freq = Xc.mean(axis=0)  # normalized in [0, 1]

#     ax.plot(x_pos, ones_freq, color="teal", linewidth=1.2)
#     ax.fill_between(x_pos, ones_freq, color="teal", alpha=0.20)
#     ax.set_title(f"Cluster {cid} (n={Xc.shape[0]})")
#     ax.set_xlabel("Position in X_bin")
#     ax.set_ylabel("ones_freq")
#     ax.set_xlim(1, L)
#     ax.set_ylim(0, 1)
#     ax.grid(True, alpha=0.25)

# for j in range(n_clusters, len(axes)):
#     axes[j].axis("off")

# fig.suptitle("ones_freq profiles per cluster (clustering on t-SNE space)", fontsize=13)
# plt.show()

# Optional: also visualize t-SNE with cluster colors
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes2[0].plot(list(k_values), sil_scores, marker="o", color="black")
axes2[0].axvline(best_k_tsne, color="red", linestyle="--", alpha=0.7, label=f"best k={best_k_tsne}")
axes2[0].set_title("Silhouette vs k (t-SNE space)")
axes2[0].set_xlabel("k")
axes2[0].set_ylabel("Silhouette")
axes2[0].grid(True, alpha=0.3)
axes2[0].legend()

sc = axes2[1].scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=labels_tsne,
    cmap="tab10",
    s=14,
    alpha=0.85,
    edgecolor="none",
)
axes2[1].set_title(f"t-SNE embedding with KMeans labels (k={best_k_tsne})")
axes2[1].set_xlabel("t-SNE 1")
axes2[1].set_ylabel("t-SNE 2")
axes2[1].grid(True, alpha=0.25)
legend2 = axes2[1].legend(*sc.legend_elements(), title="Cluster", loc="best")
axes2[1].add_artist(legend2)
plt.show()

# Save outputs for downstream usage
tsne_embedding = X_tsne
tsne_kmeans_labels = labels_tsne
tsne_best_k = best_k_tsne
tsne_silhouette_scores = sil_scores

### 9.1) Fixed-K Experiments on t-SNE and X_bin
Confronto tra configurazioni a `K` fissato: override in spazio t-SNE e clustering a `K=8` nello spazio originale `X_bin` con colorazione su t-SNE.

In [ ]:
from sklearn.decomposition import PCA

best_k_tsne = 9
kmeans_tsne = KMeans(n_clusters=best_k_tsne, random_state=None, n_init=20)
labels_tsne = kmeans_tsne.fit_predict(X_tsne)

print(f"Best k on t-SNE space (silhouette): {best_k_tsne}")
u_lbl, u_cnt = np.unique(labels_tsne, return_counts=True)
print("Cluster sizes (t-SNE space):")
for lbl, cnt in zip(u_lbl, u_cnt):
    print(f"  Cluster {lbl}: {cnt} sequences")

# -------------------------------------------------------------------------------
# 4) Figure with n-cluster subplots: ones_freq profiles
# -------------------------------------------------------------------------------
cluster_ids = np.unique(labels_tsne)
n_clusters = len(cluster_ids)
n_cols = min(3, n_clusters)
n_rows = int(np.ceil(n_clusters / n_cols))
x_pos = np.arange(1, L + 1)
n_dtm_local = int(n_dtm) if "n_dtm" in globals() else data_DTm.shape[0]

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(6 * n_cols, 3.8 * n_rows),
    constrained_layout=True,
    sharex=True,
    sharey=True,
 )
axes = np.array(axes).reshape(-1)

for ax_i, cid in enumerate(cluster_ids):
    ax = axes[ax_i]
    idx = np.where(labels_tsne == cid)[0]
    Xc = X[idx]
    ones_freq = Xc.mean(axis=0)  # normalized in [0, 1]

    person_corr = np.corrcoef(ones_freq, 1-rna_vector)[0, 1]
    dtp_frac = np.mean(idx >= n_dtm_local) if idx.size > 0 else np.nan
    ax.text(0.02, 0.95, f"corr with free sites: {person_corr:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")
    ax.text(0.02, 0.88, f"frac DTp: {dtp_frac:.3f}", transform=ax.transAxes, fontsize=10, verticalalignment="top")

    ax.plot(x_pos, ones_freq, color="teal", linewidth=1.2)
    ax.fill_between(x_pos, ones_freq, color="teal", alpha=0.20)
    ax.bar(range(len(rna_vector)), 0.1*(1- rna_vector), color="salmon", alpha=1, label="free positions")
    ax.set_title(f"Cluster {cid} (n={Xc.shape[0]})")
    ax.set_xlabel("Position in X_bin")
    ax.set_ylabel("ones_freq")
    ax.set_xlim(1, L)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)

for j in range(n_clusters, len(axes)):
    axes[j].axis("off")

fig.suptitle("ones_freq profiles per cluster (clustering on t-SNE space)", fontsize=13)
plt.show()

# Optional: also visualize t-SNE with cluster colors
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes2[0].plot(list(k_values), sil_scores, marker="o", color="black")
axes2[0].axvline(best_k_tsne, color="red", linestyle="--", alpha=0.7, label=f"best k={best_k_tsne}")
axes2[0].set_title("Silhouette vs k (t-SNE space)")
axes2[0].set_xlabel("k")
axes2[0].set_ylabel("Silhouette")
axes2[0].grid(True, alpha=0.3)
axes2[0].legend()

sc = axes2[1].scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=labels_tsne,
    cmap="tab10",
    s=14,
    alpha=0.85,
    edgecolor="none",
)
axes2[1].set_title(f"t-SNE embedding with KMeans labels (k={best_k_tsne})")
axes2[1].set_xlabel("t-SNE 1")
axes2[1].set_ylabel("t-SNE 2")
axes2[1].grid(True, alpha=0.25)
legend2 = axes2[1].legend(*sc.legend_elements(), title="Cluster", loc="best")
axes2[1].add_artist(legend2)
plt.show()

# Additional plot: PCA projection colored with t-SNE-derived cluster labels
pca_model = PCA(n_components=2, random_state=None)
X_pca_from_tsne_labels = pca_model.fit_transform(X)
pca_var = pca_model.explained_variance_ratio_

fig3, ax3 = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)
sc3 = ax3.scatter(
    X_pca_from_tsne_labels[:, 0],
    X_pca_from_tsne_labels[:, 1],
    c=labels_tsne,
    cmap="tab10",
    s=14,
    alpha=0.85,
    edgecolor="none",
)
ax3.set_title(f"PCA colored by t-SNE clustering labels (k={best_k_tsne})")
ax3.set_xlabel(f"PC1 ({pca_var[0] * 100:.1f}% var)")
ax3.set_ylabel(f"PC2 ({pca_var[1] * 100:.1f}% var)")
ax3.grid(True, alpha=0.25)
legend3 = ax3.legend(*sc3.legend_elements(), title="Cluster", loc="best")
ax3.add_artist(legend3)
plt.show()

# Save outputs for downstream usage
tsne_embedding = X_tsne
tsne_kmeans_labels = labels_tsne
tsne_best_k = best_k_tsne
tsne_silhouette_scores = sil_scores
tsne_labels_pca_embedding = X_pca_from_tsne_labels
tsne_labels_pca_model = pca_model

In [ ]:
# ===============================================================================
# FIXED-K CLUSTERING ON X_bin (K=8) + t-SNE COLORING BY THOSE CLUSTERS
# ===============================================================================

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# -------------------------------------------------------------------------------
# Prepare X_bin
# -------------------------------------------------------------------------------
if "mutation_binary_dataset" in globals():
    X_bin_local = mutation_binary_dataset
elif "X_bin" in globals():
    X_bin_local = X_bin
else:
    reference_seq = data_DN[0]
    X_bin_local = (DT_full != reference_seq).astype(np.uint8)

X = X_bin_local.astype(np.float64)
N, L = X.shape

# -------------------------------------------------------------------------------
# Clustering in ORIGINAL space (X_bin) with fixed K=8
# -------------------------------------------------------------------------------
k_fixed = 8
kmeans_xbin_k8 = KMeans(n_clusters=k_fixed, random_state=None, n_init=20)
labels_xbin_k8 = kmeans_xbin_k8.fit_predict(X)

u_lbl, u_cnt = np.unique(labels_xbin_k8, return_counts=True)
print(f"KMeans on X_bin completed with K={k_fixed}")
print("Cluster sizes:")
for lbl, cnt in zip(u_lbl, u_cnt):
    print(f"  Cluster {lbl}: {cnt} sequences")

# -------------------------------------------------------------------------------
# t-SNE embedding (compute or reuse existing with matching shape)
# -------------------------------------------------------------------------------
reuse_tsne = "tsne_embedding" in globals() and tsne_embedding.shape[0] == N
if reuse_tsne:
    X_tsne_local = tsne_embedding
    print("Using existing tsne_embedding from previous cell.")
else:
    perplexity = min(30.0, max(5.0, (N - 1) / 3.0))
    if perplexity >= N:
        perplexity = max(2.0, N - 1.0)

    tsne_local = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        random_state=None,
        max_iter=1000,
    )
    X_tsne_local = tsne_local.fit_transform(X)
    print(f"Computed new t-SNE embedding with perplexity={perplexity:.2f}")

# -------------------------------------------------------------------------------
# Plot: t-SNE colored by clustering obtained in X_bin space
# -------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(9, 7), constrained_layout=True)
sc = ax.scatter(
    X_tsne_local[:, 0],
    X_tsne_local[:, 1],
    c=labels_xbin_k8,
    cmap="tab10",
    s=14,
    alpha=0.85,
    edgecolor="none",
)
ax.set_title("t-SNE colored by KMeans labels from X_bin (K=8)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.grid(True, alpha=0.25)
legend = ax.legend(*sc.legend_elements(), title="Cluster", loc="best")
ax.add_artist(legend)
plt.show()

# Save outputs
k8_xbin_labels = labels_xbin_k8
k8_xbin_model = kmeans_xbin_k8
tsne_for_k8_plot = X_tsne_local

In [ ]:
# ===============================================================================
# K-means on DT in X_bin space: choose k minimizing
# (intra-cluster distance) / (distance from DN[0] reference)
# Then: PCA plot of best clustering + ones_freq subplot per cluster
# ===============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# -------------------------------------------------------------------------------
# Build/validate X_bin for DT against reference = first DN sequence
# -------------------------------------------------------------------------------
if "DT" not in globals():
    DT = np.concatenate((data_DTm, data_DTp), axis=0)

reference_seq = data_DN[0]

# Recompute X_bin directly from DT if absent or shape-mismatched.
if ("X_bin" not in globals()) or (X_bin.shape[0] != DT.shape[0]) or (X_bin.shape[1] != DT.shape[1]):
    X_bin_use = (DT != reference_seq).astype(np.uint8)
else:
    X_bin_use = X_bin.astype(np.uint8)

X = X_bin_use.astype(np.float64)
N, L = X.shape
n_dtm_local = int(n_dtm) if "n_dtm" in globals() else data_DTm.shape[0]

# -------------------------------------------------------------------------------
# Try multiple k values and score each by weighted mean ratio:
# ratio = mean_intra_cluster_distance / mean_distance_to_reference
# where distance_to_reference is exactly sum(X_bin) per sequence.
# -------------------------------------------------------------------------------
k_min = 2
k_max = min(15, max(2, int(np.sqrt(N)) + 2))
k_values = range(k_min, k_max + 1)

eps = 1e-12
autoK_results = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=None, n_init=20)
    labels_k = km.fit_predict(X)
    centers = km.cluster_centers_

    per_cluster = []
    weighted_score = 0.0

    for cid in range(k):
        idx = np.where(labels_k == cid)[0]
        Xc = X[idx]
        n_c = len(idx)

        # Intra-cluster: average distance to own centroid.
        intra_d = np.linalg.norm(Xc - centers[cid], axis=1).mean()

        # Requested definition: distance from reference = sum of X_bin features.
        ref_d = Xc.sum(axis=1).mean()

        ratio = intra_d / (ref_d + eps)
        weighted_score += ratio * n_c

        per_cluster.append(
            {
                "cluster": cid,
                "size": n_c,
                "intra": float(intra_d),
                "ref": float(ref_d),
                "ratio": float(ratio),
            }
        )

    weighted_score /= N

    autoK_results.append(
        {
            "k": k,
            "score": float(weighted_score),
            "labels": labels_k,
            "model": km,
            "clusters": per_cluster,
        }
    )

best_record = min(autoK_results, key=lambda r: r["score"])
autoK = best_record["k"]
autoK_labels = best_record["labels"]
autoK_model = best_record["model"]

print(f"Best k = {autoK}")
print(f"Minimum weighted ratio (intra/ref) = {best_record['score']:.6f}")
print("\nPer-cluster diagnostics (best k):")
for rec in best_record["clusters"]:
    print(
        f"  Cluster {rec['cluster']}: n={rec['size']}, "
        f"intra={rec['intra']:.4f}, ref={rec['ref']:.4f}, ratio={rec['ratio']:.6f}"
    )

# -------------------------------------------------------------------------------
# Plot 1: score vs k
# -------------------------------------------------------------------------------
ks_plot = [r["k"] for r in autoK_results]
scores_plot = [r["score"] for r in autoK_results]

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(ks_plot, scores_plot, marker="o", lw=2)
ax.axvline(autoK, color="tab:red", ls="--", lw=1.5, label=f"best k={autoK}")
ax.set_title("Model selection on X_bin: weighted intra/ref ratio")
ax.set_xlabel("k")
ax.set_ylabel("Weighted ratio (lower is better)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# -------------------------------------------------------------------------------
# Plot 2: PCA projection of best clustering
# -------------------------------------------------------------------------------
pca = PCA(n_components=2, random_state=None)
X_pca = pca.fit_transform(X)
expl_var = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
sc = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=autoK_labels,
    cmap="tab10",
    s=14,
    alpha=0.85,
    edgecolor="none",
)
ax.set_title(f"PCA of X_bin with optimal KMeans clustering (k={autoK})")
ax.set_xlabel(f"PC1 ({expl_var[0] * 100:.1f}% var)")
ax.set_ylabel(f"PC2 ({expl_var[1] * 100:.1f}% var)")
ax.grid(True, alpha=0.25)
legend = ax.legend(*sc.legend_elements(), title="Cluster", loc="best")
ax.add_artist(legend)
plt.show()

# -------------------------------------------------------------------------------
# Plot 3: ones_freq subplot for each cluster (best k)
# -------------------------------------------------------------------------------
cluster_ids = np.unique(autoK_labels)
ones_freq = np.zeros((len(cluster_ids), L), dtype=np.float64)
autoK_cluster_sizes = np.zeros(len(cluster_ids), dtype=int)
dtp_frac_per_cluster = np.zeros(len(cluster_ids), dtype=np.float64)

for i, cid in enumerate(cluster_ids):
    idx = np.where(autoK_labels == cid)[0]
    autoK_cluster_sizes[i] = len(idx)
    ones_freq[i] = X[idx].mean(axis=0)
    dtp_frac_per_cluster[i] = np.mean(idx >= n_dtm_local) if idx.size > 0 else np.nan

n_clusters = len(cluster_ids)
n_cols = min(4, n_clusters)
n_rows = int(np.ceil(n_clusters / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.4 * n_cols, 2.7 * n_rows),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
axes = np.array(axes).reshape(-1)

x_pos = np.arange(L)

for ax_i, cid in enumerate(cluster_ids):
    ax = axes[ax_i]
    ax.plot(x_pos, ones_freq[ax_i], lw=1.2, color="tab:blue")
    ax.fill_between(x_pos, 0.0, ones_freq[ax_i], alpha=0.18, color="tab:blue")
    ax.text(0.02, 0.95, f"frac DTp: {dtp_frac_per_cluster[ax_i]:.3f}", transform=ax.transAxes, fontsize=9, verticalalignment="top")
    ax.set_title(f"Cluster {cid} (n={autoK_cluster_sizes[ax_i]})")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)

for j in range(n_clusters, len(axes)):
    axes[j].axis("off")

fig.suptitle("ones_freq per cluster (optimal k)", y=1.02)
for ax in axes[:n_clusters]:
    ax.set_xlabel("Position")
    ax.set_ylabel("ones_freq")

plt.show()

# Save outputs for downstream cells
autoK_best = best_record
autoK_freq_ones = ones_freq

In [ ]:
DT = np.concatenate((data_DTm, data_DTp), axis=0)
rng = np.random.default_rng()
idx = rng.choice(DT.shape[0], size=14000, replace=False)
DT_subsample = DT[idx, :]
print(f"DT_subsample shape: {DT_subsample.shape}")

## 9) Additional Reference-Based Analyses and Diagnostics
Analisi complementari rispetto alla reference e controlli diagnostici finali.

In [ ]:

# ===============================================================================
# UNION MASK ANALYSIS: reference = first sequence of DN
# ===============================================================================
# For each sequence in DT_subsample, compute the union mask with the reference
# sequence data_DN[0]. Since both clusters are singletons, the union mask is just
# the set of positions where the two sequences differ.

reference_seq = data_DN[0]

if DT_subsample.shape[1] != reference_seq.shape[0]:
    raise ValueError(
        f"Length mismatch: DT_subsample has L={DT_subsample.shape[1]}, "
        f"reference has L={reference_seq.shape[0]}"
    )

union_masks_vs_ref = DT_subsample != reference_seq
mutation_counts_per_position = union_masks_vs_ref.sum(axis=0)
mutation_fraction_per_position = mutation_counts_per_position / DT_subsample.shape[0]
mask_sizes_per_sequence = union_masks_vs_ref.sum(axis=1)

print(f"Reference sequence taken from data_DN[0]")
print(f"DT_subsample shape: {DT_subsample.shape}")
print(f"Average union-mask size vs reference: {mask_sizes_per_sequence.mean():.2f}")
print(f"Min/Max union-mask size vs reference: {mask_sizes_per_sequence.min()} / {mask_sizes_per_sequence.max()}")
print(f"Positions mutated in at least one DT_subsample sequence: {(mutation_counts_per_position > 0).sum()} / {DT_subsample.shape[1]}")

fig, axes = plt.subplots(2, 1, figsize=(16, 10), constrained_layout=True)

# Position-by-position mutation counts
positions_idx = np.arange(1, DT_subsample.shape[1] + 1)
axes[0].bar(positions_idx, mutation_counts_per_position, color="steelblue", edgecolor="black", linewidth=0.4)
axes[0].set_title("Mutation count per position vs DN reference", fontsize=16)
axes[0].set_xlabel("Alignment position", fontsize=13)
axes[0].set_ylabel("Number of DT_subsample sequences", fontsize=13)
axes[0].grid(True, axis="y", alpha=0.3)
axes[0].set_xlim(0.5, DT_subsample.shape[1] + 0.5)

# Distribution of union-mask sizes per sequence

axes[1].hist(mask_sizes_per_sequence, bins=500, color="indianred", edgecolor="black", alpha=0.85)
axes[1].set_title("Distribution of union-mask sizes vs DN reference", fontsize=16)
axes[1].set_xlabel("Union-mask size", fontsize=13)
axes[1].set_ylabel("Number of sequences", fontsize=13)
axes[1].grid(True, axis="y", alpha=0.3)

plt.show()

# Top variable positions
top_k = 15
top_positions = np.argsort(mutation_counts_per_position)[::-1][:top_k]
print("\nTop positions by mutation count vs reference:")
for rank, pos in enumerate(top_positions, start=1):
    print(
        f"{rank:2d}. Position {pos + 1:3d}: "
        f"{mutation_counts_per_position[pos]:4d} sequences "
        f"({100 * mutation_fraction_per_position[pos]:5.1f}%)"
    )


In [ ]:
# ===============================================================================
# PATTERN ANALYSIS OF MUTATIONS VS REFERENCE
# ===============================================================================
# Each row of union_masks_vs_ref is a binary mutation pattern relative to data_DN[0].
# Here we identify which exact patterns occur, how often they occur, and visualize
# the most frequent ones.

patterns, counts = np.unique(union_masks_vs_ref, axis=0, return_counts=True)
order = np.argsort(counts)[::-1]
patterns = patterns[order]
counts = counts[order]
pattern_sizes = patterns.sum(axis=1)
pattern_frequencies = counts / counts.sum()

print(f"Number of distinct mutation patterns: {len(patterns)}")
print(f"Most frequent pattern count: {counts[0]} sequences ({100 * pattern_frequencies[0]:.2f}%)")
print(f"Top 10 patterns cover {100 * pattern_frequencies[:10].sum():.2f}% of DT_subsample")

print("\nTop mutation patterns vs reference:")
top_n = min(10, len(patterns))
for rank in range(top_n):
    mutated_positions = np.flatnonzero(patterns[rank]) + 1
    preview = ", ".join(map(str, mutated_positions[:12]))
    if len(mutated_positions) > 12:
        preview += ", ..."
    print(
        f"{rank + 1:2d}. count = {counts[rank]:5d} ({100 * pattern_frequencies[rank]:5.2f}%) | "
        f"mask size = {pattern_sizes[rank]:2d} | positions: [{preview}]"
    )

fig, axes = plt.subplots(2, 1, figsize=(18, 10), constrained_layout=True)

# Bar chart of the most frequent patterns
pattern_labels = [f"P{idx+1}" for idx in range(top_n)]
axes[0].bar(pattern_labels, counts[:top_n], color="darkslateblue", edgecolor="black", alpha=0.85)
axes[0].set_title("Frequency of the most common mutation patterns", fontsize=16)
axes[0].set_xlabel("Pattern rank", fontsize=13)
axes[0].set_ylabel("Number of sequences", fontsize=13)
axes[0].grid(True, axis="y", alpha=0.3)

# Heatmap of the top patterns across positions
im = axes[1].imshow(patterns[:top_n].astype(int), aspect="auto", cmap="Greys", interpolation="nearest")
axes[1].set_title("Binary mutation patterns of the most common masks", fontsize=16)
axes[1].set_xlabel("Alignment position", fontsize=13)
axes[1].set_ylabel("Pattern rank", fontsize=13)
axes[1].set_yticks(np.arange(top_n))
axes[1].set_yticklabels(pattern_labels)

cbar = fig.colorbar(im, ax=axes[1], fraction=0.02, pad=0.01)
cbar.set_label("Mutation vs reference (0/1)", fontsize=12)

plt.show()

In [ ]:
# Verifica teorica/pratica sul range dei costi iniziali (singleton -> singleton)
# Per singleton, cost(C_i, C_j) = numero di posizioni diverse tra le due sequenze.
# Quindi il costo e sempre in [0, L], e puo valere esattamente L.

X = DT_subsample
Lx = X.shape[1]

pair_cost = (X[:, None, :] != X[None, :, :]).sum(axis=2)
upper = pair_cost[np.triu_indices(X.shape[0], 1)]

print(f"L = {Lx}")
print(f"Costi singleton: min={upper.min()}, max={upper.max()}, media={upper.mean():.2f}")
print(f"Numero di coppie con costo = L ({Lx}): {(upper == Lx).sum()} su {upper.size}")

# Se questo assert fallisce, allora c'e un bug nella logica dei costi.
assert upper.max() <= Lx, "Trovato costo > L: non dovrebbe mai succedere."

In [ ]:
# Quick diagnostic: initial singleton merge costs should not be all zero
X = DT_subsample
n = X.shape[0]

pair_cost = (X[:, None, :] != X[None, :, :]).sum(axis=2)
upper = pair_cost[np.triu_indices(n, 1)]

print(f"Initial directed/symmetric singleton costs -> min: {upper.min()}, max: {upper.max()}, zeros: {(upper == 0).sum()} / {upper.size}")